# Notebook 3: Loss Function and Scalar Backpropagation
## Neural Network from Scratch: Mathematical Derivation and Explicit-Loop Implementation
In the previous notebook, we constructed the forward pass of a one-hidden-layer neural network using scalar calculations and explicit Python loops.
The network maps an input vector
$
\mathbf{x}\in\mathbb{R}^{d}
$
through a hidden layer containing (h) neurons and produces a scalar probability for binary classification.
In this notebook, we derive and implement the corresponding backward pass.

The central question is:
Given the prediction error produced by the network, how does each individual weight and bias contribute to that error?

We will answer this question by applying the chain rule one derivative at a time.

### Objectives
By the end of this notebook, we will:
1. Define binary cross-entropy loss.
2. Derive the output-layer error term.
3. Backpropagate the error into the hidden layer.
4. Derive every weight and bias gradient.
5. Implement backpropagation using explicit loops.
6. Manually verify the gradients on a small synthetic network.
7. Perform numerical gradient checking.
8. Apply one complete forward and backward pass to an MNIST image.

This notebook deliberately avoids vectorized backpropagation. Vectorization will be introduced only after the scalar calculations have been verified.


## 1. Network Architecture and Notation
We use a one-hidden-layer binary classification network.
For an input vector with (d) features and a hidden layer with (h) neurons, the parameter dimensions are:
$$
W^{(1)}\in\mathbb{R}^{h\times d}
$$
$$
\mathbf{b}^{(1)}\in\mathbb{R}^{h}
$$
$$
\mathbf{w}^{(2)}\in\mathbb{R}^{h}
$$
$$
b^{(2)}\in\mathbb{R}
$$
The first-layer weight $W^{(1)}_{ji}$ connects input feature $x_i$ to hidden neuron (j).
The second-layer weight $w^{(2)}_j$ connects hidden neuron (j) to the output neuron.
For each hidden neuron (j):
$$
\sum_{i=1}^{d}W_{ji}^{(1)}x_i+b_j^{(1)}
$$
$$
\sigma\left(z_j^{(1)}\right)
$$
The output pre-activation is:
$$
\sum_{j=1}^{h}w_j^{(2)}a_j^{(1)}+b^{(2)}
$$
The predicted probability is:
$$
a^{(2)}
=\sigma\left(z^{(2)}\right)
$$
where the sigmoid function is:
$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$
The model predicts the probability that the input belongs to class (1).

In [7]:
import math
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=8, suppress=True)

%load_ext autoreload
%autoreload 2

from neural_network_functions import (
    sigmoid,
    hidden_layer_forward_loop,
    output_layer_forward_loop,
    forward_propagation_loop
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Scalar Sigmoid Function and Its Derivative
The sigmoid activation function is:
$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$
Its derivative is:
$$
\sigma(z)\left(1-\sigma(z)\right)
$$
Because the activation value $a=\sigma(z)$ is already calculated during forward propagation, we can write the derivative as:
$$
\sigma'(z)=a(1-a)
$$
This form avoids recalculating the exponential function during backpropagation.

In [8]:
def sigmoid_scalar(z):
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))

    exp_z = math.exp(z)
    return exp_z / (1.0 + exp_z)
    
def sigmoid_derivative_from_activation(a):
    return a * (1.0 - a)

In [9]:
test_values = [-3.0, 0.0, 3.0]
for z in test_values:
    activation = sigmoid_scalar(z)
    derivative = sigmoid_derivative_from_activation(activation)
    print(
        f"z = {z:5.1f}, "
        f"sigmoid(z) = {activation:.6f}, "
        f"sigmoid'(z) = {derivative:.6f}"
    )

z =  -3.0, sigmoid(z) = 0.047426, sigmoid'(z) = 0.045177
z =   0.0, sigmoid(z) = 0.500000, sigmoid'(z) = 0.250000
z =   3.0, sigmoid(z) = 0.952574, sigmoid'(z) = 0.045177


### Analysis
The sigmoid derivative is largest when $z=0$, where:
$$
\sigma(0)=0.5
$$
and:
$$
\sigma'(0)=0.5(1-0.5)=0.25
$$
The derivative becomes small when the activation approaches either (0) or (1). This behavior will later help explain the vanishing-gradient problem in deeper sigmoid networks.


## 3. Binary Cross-Entropy Loss
For binary classification, the true label satisfies:
$$
y\in{0,1}
$$
and the network produces:
$$
\hat{y}\in(0,1)
$$
The binary cross-entropy loss for one observation is:
$$
-\left[
y\log(\hat{y})
+
(1-y)\log(1-\hat{y})
\right]
$$
The formula contains two cases.

Case 1: $y=1$
$$
\mathcal{L}(1,\hat{y})=-\log(\hat{y})
$$
The loss becomes small when $\hat{y}$ is close to (1).

Case 2: $y=0$
$$
\mathcal{L}(0,\hat{y})=-\log(1-\hat{y})
$$
The loss becomes small when $\hat{y}$ is close to (0).
Binary cross-entropy therefore penalizes confident incorrect predictions much more strongly than uncertain predictions.

In [10]:
def binary_cross_entropy_scalar(y, y_hat, epsilon=1e-12):
    y_hat_clipped = min(max(y_hat, epsilon), 1.0 - epsilon)
    bce = -(y * math.log(y_hat_clipped) + (1.0 - y) * math.log(1.0 - y_hat_clipped))
    
    return bce

In [11]:
examples = [
(1.0, 0.99),
(1.0, 0.70),
(1.0, 0.10),
(0.0, 0.01),
(0.0, 0.30),
(0.0, 0.90),
]

for y, y_hat in examples:
    loss = binary_cross_entropy_scalar(y, y_hat)
    print(
        f"y = {y:.0f}, "
        f"y_hat = {y_hat:.2f}, "
        f"loss = {loss:.6f}"
    )

y = 1, y_hat = 0.99, loss = 0.010050
y = 1, y_hat = 0.70, loss = 0.356675
y = 1, y_hat = 0.10, loss = 2.302585
y = 0, y_hat = 0.01, loss = 0.010050
y = 0, y_hat = 0.30, loss = 0.356675
y = 0, y_hat = 0.90, loss = 2.302585


### Analysis
When the true class is (1), increasing $\hat{y}$ decreases the loss.
When the true class is (0), decreasing $\hat{y}$ decreases the loss.
A confidently incorrect prediction such as:
$$
y=1,\qquad \hat{y}=0.10
$$
produces a much larger loss than a less confident prediction such as:
$$
y=1,\qquad \hat{y}=0.70
$$
This makes binary cross-entropy especially appropriate for probability-based binary classification.

## 4. Deriving the Output-Layer Gradient
The output neuron performs two operations:
$$
\sum_{j=1}^{h}w_j^{(2)}a_j^{(1)}+b^{(2)}
$$
$$
\hat{y}=\sigma\left(z^{(2)}\right)
$$
The loss is:
$$
-\left[
y\log(\hat{y})
+
(1-y)\log(1-\hat{y})
\right]
$$
To update the network, we first need:
$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}
$$
Using the chain rule:
$$
\frac{\partial\mathcal{L}}{\partial\hat{y}}
\frac{\partial\hat{y}}{\partial z^{(2)}}
$$
First, differentiate the loss with respect to $\hat{y}$:
$$
-\frac{y}{\hat{y}}
+
\frac{1-y}{1-\hat{y}}
$$
The derivative of the sigmoid output is:
$$
\hat{y}(1-\hat{y})
$$
Multiplying the two expressions gives:
$$
\left(
-\frac{y}{\hat{y}}
+
\frac{1-y}{1-\hat{y}}
\right)
\hat{y}(1-\hat{y})
$$
After simplification:
$$
\hat{y}-y
$$
We define the output-layer error term as:
$$
\boxed{
\delta^{(2)}=\hat{y}-y
}
$$
This simplification occurs specifically because binary cross-entropy is combined with a sigmoid output.

## 5. Output-Layer Weight and Bias Gradients
The output pre-activation is:
$$
\sum_{j=1}^{h}w_j^{(2)}a_j^{(1)}+b^{(2)}
$$
For output weight $w_j^{(2)}$:
$$
a_j^{(1)}
$$
Using the chain rule:
$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}
\frac{\partial z^{(2)}}{\partial w_j^{(2)}}
$$
Therefore:
$$
\delta^{(2)}a_j^{(1)}
$$

For the output bias:
$$
\frac{\partial z^{(2)}}{\partial b^{(2)}}=1
$$
Therefore:
$$
\delta^{(2)}
$$

The output-layer gradients are therefore:
$$
\frac{\partial\mathcal{L}}{\partial w_j^{(2)}}=
(\hat{y}-y)a_j^{(1)}
$$
$$
\frac{\partial\mathcal{L}}{\partial b^{(2)}}=
\hat{y}-y
$$

## 6. Backpropagating into the Hidden Layer
Hidden neuron (j) influences the loss through the following path:
$$
z_j^{(1)}
\rightarrow
a_j^{(1)}
\rightarrow
z^{(2)}
\rightarrow
\hat{y}
\rightarrow
\mathcal{L}
$$
We need:
$$
\frac{\partial\mathcal{L}}{\partial z_j^{(1)}}
$$
Applying the chain rule:
$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}
\frac{\partial z^{(2)}}{\partial a_j^{(1)}}
\frac{\partial a_j^{(1)}}{\partial z_j^{(1)}}
$$
We already know:
$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}=
\delta^{(2)}
$$
Because:
$$
z^{(2)}=
\sum_{j=1}^{h}w_j^{(2)}a_j^{(1)}+b^{(2)}
$$
we have:
$$
\frac{\partial z^{(2)}}{\partial a_j^{(1)}}=w_j^{(2)}
$$
The hidden activation is sigmoid, so:
$$
\frac{\partial a_j^{(1)}}{\partial z_j^{(1)}}=
a_j^{(1)}\left(1-a_j^{(1)}\right)
$$
Combining these terms:
$$
\delta^{(2)}
w_j^{(2)}
a_j^{(1)}
\left(1-a_j^{(1)}\right)
$$
We define the hidden-layer error term:
$$
\delta^{(1)}=
\delta^{(2)}
w_j^{(2)}
a_j^{(1)}
\left(1-a_j^{(1)}\right)
$$
This term measures how much hidden neuron (j)'s pre-activation contributed to the final loss.

## 7. Hidden-Layer Weight and Bias Gradients
The pre-activation of hidden neuron (j) is:
$$
\sum_{i=1}^{d}W_{ji}^{(1)}x_i+b_j^{(1)}
$$
For the weight connecting input (i) to hidden neuron (j):
$$
x_i
$$
Therefore:
$$
\frac{\partial\mathcal{L}}{\partial z_j^{(1)}}
\frac{\partial z_j^{(1)}}{\partial W_{ji}^{(1)}}
$$
which gives:
$$
\delta_j^{(1)}x_i
$$
For the hidden bias:
$$
\frac{\partial z_j^{(1)}}{\partial b_j^{(1)}}=1
$$
Therefore:
$$
\delta_j^{(1)}
$$

The full scalar backpropagation equations are:
$$
\delta^{(2)}=\hat{y}-y
$$
$$
\frac{\partial\mathcal{L}}{\partial w_j^{(2)}}=
\delta^{(2)}a_j^{(1)}
$$
$$
\frac{\partial\mathcal{L}}{\partial b^{(2)}}=
\delta^{(2)}
$$
$$
\delta^{(1)}=
\delta^{(2)}
w_j^{(2)}
a_j^{(1)}
\left(1-a_j^{(1)}\right)
$$
$$
\frac{\partial\mathcal{L}}{\partial W_{ji}^{(1)}}=
\delta_j^{(1)}x_i
$$
$$
\frac{\partial\mathcal{L}}{\partial b_j^{(1)}}=
\delta_j^{(1)}
$$


# Cell 16 — Markdown
## 8. Explicit-Loop Forward Propagation
We first reproduce the scalar forward pass.
No matrix multiplication is used in this implementation. Every weighted sum is calculated with explicit Python loops.
The function returns both the prediction and a cache containing the intermediate values required during backpropagation.

In [12]:
def forward_scalar(x, W1, b1, W2, b2):
    """
    x : array of shape (n_inputs,)
    W1 : array of shape (n_hidden, n_inputs)
    b1 : array of shape (n_hidden,)
    W2 : array of shape (n_hidden,)
    b2 : scalar

    Returns
    y_hat : scalar predicted probability
    cache : dictionary containing intermediate calculations
    """
    cache = forward_propagation_loop(x, W1, b1, W2, b2)
    y_hat = cache["y_hat"]

    return y_hat, cache

## 9. Explicit-Loop Backpropagation
The backward function receives:
the true label (y);
the second-layer weights;
the cache produced during the forward pass.
It calculates:
$$
\frac{\partial\mathcal{L}}{\partial W^{(1)}},
\quad
\frac{\partial\mathcal{L}}{\partial \mathbf{b}^{(1)}},
\quad
\frac{\partial\mathcal{L}}{\partial \mathbf{w}^{(2)}},
\quad
\frac{\partial\mathcal{L}}{\partial b^{(2)}}
$$
Each gradient is calculated using explicit loops.
An important implementation detail is that the hidden-layer error must use the current output weights (W^{(2)}). The weights should not be updated until every gradient has been calculated.

In [13]:
# For one observation
def backward_scalar(y, W2, cache):
    """
    y : scalar true label
    W2 : array of shape (n_hidden,)
    cache : intermediate values from forward_scalar

    Returns
    gradients : dictionary containing all parameter gradients
    """
    x = cache["x"]
    a1 = cache["a1"]
    y_hat = cache["y_hat"]

    n_inputs = len(x)
    n_hidden = len(a1)

    dW1 = np.zeros((n_hidden, n_inputs))
    db1 = np.zeros(n_hidden)
    dW2 = np.zeros(n_hidden)

    delta1 = np.zeros(n_hidden)

    # Output-layer error
    delta2 = y_hat - y

    # Output-layer gradients
    for j in range(n_hidden):
        dW2[j] = delta2 * a1[j]

    db2 = delta2

    # Hidden-layer errors
    for j in range(n_hidden):
        sigmoid_derivative = a1[j] * (1.0 - a1[j])

        delta1[j] = (
            delta2
            * W2[j]
            * sigmoid_derivative
        )

    # Hidden-layer gradients
    for j in range(n_hidden):
        for i in range(n_inputs):
            dW1[j, i] = delta1[j] * x[i]

        db1[j] = delta1[j]

    gradients = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2,
        "delta1": delta1,
        "delta2": delta2,
    }

    return gradients

## 10. Manual Gradient Verification on a Tiny Network
We now test the derivation on a network containing:
two input features;
two hidden neurons;
one output neuron.

The input is:
$$
\begin{bmatrix}
0.6\
-0.2
\end{bmatrix}
$$
The true label is:
$
y=1
$

The first-layer parameters are:
$$
\begin{bmatrix}
0.1 & -0.3\\
0.4 & 0.2
\end{bmatrix}
$$
$$
\begin{bmatrix}
0.0\
0.1
\end{bmatrix}
$$
The output-layer parameters are:
$$
\begin{bmatrix}
0.2\
-0.5
\end{bmatrix}
$$
$$
b^{(2)}=-0.1
$$
Because the network is very small, we can inspect every forward and backward calculation.


### Explicit-Loop Forward Propagation

In [14]:
x_tiny = np.array([0.6, -0.2])
y_tiny = 1.0
W1_tiny = np.array([
[0.1, -0.3],
[0.4, 0.2],
])
b1_tiny = np.array([0.0, 0.1])
W2_tiny = np.array([0.2, -0.5])
b2_tiny = -0.1

y_hat_tiny, cache_tiny = forward_scalar(
    x=x_tiny,
    W1=W1_tiny,
    b1=b1_tiny,
    W2=W2_tiny,
    b2=b2_tiny,
)

loss_tiny = binary_cross_entropy_scalar(
    y=y_tiny,
    y_hat=y_hat_tiny,
)

print("Hidden pre-activations z1:")
print(cache_tiny["z1"])
print("\nHidden activations a1:")
print(cache_tiny["a1"])
print("\nOutput pre-activation z2:")
print(cache_tiny["z2"])
print("\nPredicted probability:")
print(y_hat_tiny)
print("\nBinary cross-entropy loss:")
print(loss_tiny)

Hidden pre-activations z1:
[0.12 0.3 ]

Hidden activations a1:
[0.52996405 0.57444252]

Output pre-activation z2:
-0.28122844805291514

Predicted probability:
0.4301526314201492

Binary cross-entropy loss:
0.8436151764857425


### Manual Forward-Pass Calculation
For hidden neuron 1:
$$
(0.1)(0.6)+(-0.3)(-0.2)+0
$$
$$
z_1^{(1)}=0.12
$$
$$
a_1^{(1)}=\sigma(0.12)\approx0.529964
$$
For hidden neuron 2:
$$
(0.4)(0.6)+(0.2)(-0.2)+0.1
$$
$$
z_2^{(1)}=0.30
$$
$$
a_2^{(1)}=\sigma(0.30)\approx0.574443
$$
The output pre-activation is:
$$
(0.2)(0.529964)
+
(-0.5)(0.574443)
-0.1
$$
$$
z^{(2)}\approx-0.281228
$$
Therefore:
$$
\sigma(-0.281228)
\approx0.430153
$$
The loss is:
$$
-\log(0.430153)
\approx0.843615
$$

### Explicit-Loop Backpropagation

In [15]:
gradients_tiny = backward_scalar(
    y=y_tiny,
    W2=W2_tiny,
    cache=cache_tiny,
)

print("Output error delta2:")
print(gradients_tiny["delta2"])
print("\nOutput-weight gradients dW2:")
print(gradients_tiny["dW2"])
print("\nOutput-bias gradient db2:")
print(gradients_tiny["db2"])
print("\nHidden errors delta1:")
print(gradients_tiny["delta1"])
print("\nHidden-weight gradients dW1:")
print(gradients_tiny["dW1"])
print("\nHidden-bias gradients db1:")
print(gradients_tiny["db1"])

Output error delta2:
-0.5698473685798509

Output-weight gradients dW2:
[-0.30199862 -0.32734456]

Output-bias gradient db2:
-0.5698473685798509

Hidden errors delta1:
[-0.02839004  0.06965196]

Hidden-weight gradients dW1:
[[-0.01703402  0.00567801]
 [ 0.04179118 -0.01393039]]

Hidden-bias gradients db1:
[-0.02839004  0.06965196]


### Manual Backward-Pass Calculation
The output error is:
$$
\hat{y}-y=
0.430153-1
$$

$$
\boxed{\delta^{(2)}\approx-0.569847}
$$
The output-weight gradients are:
$$
\delta^{(2)}a_1^{(1)}=
(-0.569847)(0.529964)
\approx-0.301999
$$
Similarly:
$$
(-0.569847)(0.574443)
\approx-0.327345
$$
The output-bias gradient is:
$$
-0.569847
$$
For hidden neuron 1:
$$
\delta^{(2)}
w_1^{(2)}
a_1^{(1)}
\left(1-a_1^{(1)}\right)=
(-0.569847)(0.2)(0.529964)(1-0.529964)
$$
$$
\boxed{
\delta_1^{(1)}\approx-0.028390
}
$$
For hidden neuron 2:
$$
(-0.569847)(-0.5)(0.574443)(1-0.574443)
$$
$$
\boxed{
\delta_2^{(1)}\approx0.069652
}
$$
The first hidden neuron's weight gradients are:
$$
(-0.028390)(0.6)
\approx-0.017034
$$
$$
(-0.028390)(-0.2)
\approx0.005678
$$
The second hidden neuron's weight gradients are:
$$
(0.069652)(0.6)
\approx0.041791
$$
$$
(0.069652)(-0.2)
\approx-0.013930
$$

The first hidden neuron's bias gradients are:
$$
-0.028390
$$
The second hidden neuron's bias gradients are:
$$
0.069652
$$

These results should match the values returned by backward_scalar.
